# 03 - Job Recommender (Unsupervised)
Core Engine: TF-IDF vectorization of the job corpus + cosine similarity against a resume. This is the heart of the SmartHire portal's job search/matching feature.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
from src.models.recommender import fit_job_vectorizer, recommend_jobs, precision_at_k

vectorizer, matrix = fit_job_vectorizer()

Fitted job TF-IDF: 69991 jobs x 15000 features
Saved -> /home/claude/smarthire/models/job_tfidf_vectorizer.pkl, /home/claude/smarthire/models/job_tfidf_matrix.pkl


## Try it on a few sample resumes

In [2]:
resumes = pd.read_parquet('../data/processed/resumes_clean.parquet')
sample = resumes.iloc[0]
print('Resume category:', sample['category'])
recommend_jobs(sample['resume_text_clean'], top_n=10)

Resume category: Hr


,job_id,title,company,location,skills,experience,source,match_score
0,44911,Human Resources Manager,MSI Recruiting,"Boynton Beach, FL",Human Resources,Mid-Senior level,linkedin,0.216233
1,13857,Manager HR &amp; Administration,Netware Softtech Solution,"Salem, India",,,naukri,0.196422
2,65505,Human Resources Manager,,"New York, NY",Human Resources,,linkedin,0.175337
3,39094,City Manager,"Greyhound Lines, Inc.","Boston, MA",Management,Mid-Senior level,linkedin,0.174224
4,54198,"EXECUTIVE DIRECTOR, HRA PERFORMANCE & REPORTING",NYC Department of Social Services,"Manhattan, NY","Analyst, Information Technology, Research",Director,linkedin,0.170148
5,30979,Human Resources Business Partner,The JPI Group,Nashville Metropolitan Area,"Consulting, Management, Strategy/Planning",Mid-Senior level,linkedin,0.169860
6,11513,"Manager HR ( Female Only): ""night Shift"" at Noida",Compunnel Technology India Private Limited,"Noida, India",,,naukri,0.169304
7,36110,Senior HR and Payroll Business Partner,Univest,"Lansdale, PA",Human Resources,Mid-Senior level,linkedin,0.168578
8,57817,Resort Operations Manager,Hyatt Hotels Corporation,"Lahaina, HI","Management, Manufacturing",Director,linkedin,0.167922
9,60093,Business Administrator,City of Pittsburgh,"Pittsburgh, PA","Accounting/Auditing, Administrative, Strategy/...",Associate,linkedin,0.166547


In [3]:
sample2 = resumes[resumes['category']=='Information-Technology'].iloc[0]
recommend_jobs(sample2['resume_text_clean'], top_n=10)

,job_id,title,company,location,skills,experience,source,match_score
0,50141,Linux System Administrator,Super Systems Inc (SSI),"Silver Spring, MD",Information Technology,,linkedin,0.220657
1,62472,Senior System Administrator,Comrise,"Washington, DC",Information Technology,Mid-Senior level,linkedin,0.217390
2,55295,Senior System Administrator,Comrise,"New York, NY",Information Technology,Mid-Senior level,linkedin,0.215493
3,17644,Senior Manager/manager - IT Infrastructure,Antrors HR Solutions,"Delhi, India",,,naukri,0.213893
4,3695,Manager- Technology,Microland Limited,"Bengaluru, India",,,naukri,0.204793
5,50970,Senior Desktop Engineer,Mitchell Martin Inc.,"Pennsylvania, United States",Engineering,Mid-Senior level,linkedin,0.203573
6,18924,Executive Briefing Program,Apptio INC,"Bengaluru, India",,,naukri,0.202720
7,53986,Senior Cloud Engineer,InfoServ LLC,United States,"Advertising, Analyst, Consulting",Mid-Senior level,linkedin,0.200204
8,44142,Systems Administrator/Engineer,hackajob,"Plano, TX",Information Technology,Mid-Senior level,linkedin,0.198233
9,51127,Network Engineer,MSys Inc.,"Jackson, MS",Information Technology,,linkedin,0.197770


## Quantitative check: Precision@K
The job corpus has no ground-truth category label, so we use a keyword-overlap proxy: of the top-K recommended jobs, how many have a title that literally shares a token with the resume's known category name? This *undercounts* true positives (e.g. category 'Hr' vs title 'Human Resources Manager' share no exact token) so treat it as a conservative lower bound, not an absolute score -- paired with the qualitative spot-checks above for the full picture.

In [4]:
import numpy as np
sample_resumes = resumes.groupby('category').head(3).sample(40, random_state=42)
scores = [precision_at_k(r['category'], r['resume_text_clean'], k=10) for _, r in sample_resumes.iterrows()]
print(f'Mean Precision@10 (keyword-overlap proxy) across {len(scores)} resumes: {np.mean(scores):.3f}')

Mean Precision@10 (keyword-overlap proxy) across 40 resumes: 0.212


## Takeaways
- Qualitatively, recommendations are strong: an HR resume surfaces HR/People-Ops management roles, an IT resume surfaces developer/engineer roles.
- The Precision@10 proxy metric is intentionally strict and undercounts true matches -- a better evaluation would need human-labeled relevance judgments, which is out of scope for this project.
- TF-IDF + cosine similarity is fast (~milliseconds per query on 70k jobs) and interpretable, making it a good fit for a content-based recommender at this scale.